In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter
import pyttsx3  # for voice feedback

# ---------------- CONFIG ----------------
# NOTE: Make sure this path is correct on your system
DATA_ROOT = r"C:\Users\harish\Downloads\dataset\unified_db\processed_labeled_v3"
OUT_ROOT = os.path.join(os.path.dirname(DATA_ROOT), "dgcnn_simplified_results")
os.makedirs(OUT_ROOT, exist_ok=True)

NUM_POINTS = 1024
BATCH_SIZE = 16  # You can try reducing this to 8 or 4 if DGCNN uses more VRAM
EPOCHS = 100
# --- CRITICAL ---: Set to 3 (XYZ).
INPUT_FEATURES = 3
LR = 1e-4
NUM_VIS_SAMPLES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# --- We will ONLY train on these classes. Everything else will be ignored.
CLASSES_TO_USE = [
    "wall", "floor", "cabinet", "chair", "table", "door", "window",
    "road", "sidewalk", "building", "vegetation", "car"
]
# Index 0 will be "unknown" and ignored.

# ---------------- Object Mapping ----------------
RAW_IDX_TO_NAME = {
    0:"wall",1:"floor",2:"cabinet",3:"bed",4:"chair",5:"sofa",6:"table",7:"door",
    8:"window",9:"bookshelf",10:"picture",11:"counter",12:"desk",13:"curtain",
    14:"refrigerator",15:"shower curtain",16:"toilet",17:"sink",18:"bathtub",19:"other furniture",
    20:"road",21:"sidewalk",22:"building",23:"traffic light",24:"traffic sign",
    25:"vegetation",26:"terrain",27:"car",28:"truck",29:"bus",30:"person",
    31:"bicycle",32:"motorcycle",33:"animal",34:"pole",35:"fence",36:"sky",37:"unknown"
}
danger_objs = {"person", "car", "truck", "bus", "motorcycle", "bicycle"}

# ---------------- Utilities ----------------
def find_npz_files(root):
    files = []
    for r, _, fnames in os.walk(root):
        for f in fnames:
            if f.lower().endswith(".npz"):
                files.append(os.path.join(r, f))
    return sorted(files)

def safe_load_npz(fp):
    data = np.load(fp, allow_pickle=True)
    pts = np.asarray(data["points_ds"]).astype(np.float32)
    sem = data["semseg_labels"] if "semseg_labels" in data else None
    if sem is not None:
        sem = np.asarray(sem).reshape(-1)
    return pts, sem

def align_labels(points, sem):
    N = points.shape[0]
    if sem is None:
        return np.zeros(N, dtype=np.int64)
    sem = sem[:N] if sem.shape[0] >= N else np.pad(sem, (0, N-len(sem)))
    return sem.astype(np.int64)

def calculate_class_weights(dataloader, num_classes):
    counts = np.zeros(num_classes)
    for _, labels in tqdm(dataloader, desc="Calculating class weights"):
        labels = labels.reshape(-1)
        # We must ignore index 0 (unknown) in weight calculation
        counts += np.bincount(labels[labels != 0], minlength=num_classes)
    
    counts[0] = 1.0 
    total_points = np.sum(counts[1:])
    epsilon = 1e-6  
    
    class_weights = total_points / (counts * num_classes + epsilon)
    class_weights[0] = 1.0
    
    class_weights = np.clip(class_weights, a_min=0.1, a_max=1000)  
    return torch.from_numpy(class_weights).float()

# ---------------- Dataset ----------------
class NPZSegDataset(Dataset):
    def __init__(self, files, num_points=NUM_POINTS, augment=False):
        self.files = files
        self.num_points = num_points
        self.augment = augment
        
        self.final_name_to_idx = {name: i + 1 for i, name in enumerate(sorted(CLASSES_TO_USE))}
        self.final_name_to_idx["unknown"] = 0
        self.num_classes = len(self.final_name_to_idx)
        
        self.raw_to_final_idx = {}
        for raw_idx, name in RAW_IDX_TO_NAME.items():
            self.raw_to_final_idx[raw_idx] = self.final_name_to_idx.get(name, 0)
            
        self.final_idx_to_name = {idx: name for name, idx in self.final_name_to_idx.items()}

    def __len__(self):
        return len(self.files)

    def _augment(self, pts):
        xyz = pts[:, :3]
        
        theta = np.random.uniform(0, 2*np.pi)
        R = np.array([[np.cos(theta), -np.sin(theta), 0],
                      [np.sin(theta),  np.cos(theta), 0],
                      [0,0,1]], dtype=np.float32)
        xyz = xyz @ R.T
        
        if np.random.rand() > 0.5:
             xyz[:, 0] = -xyz[:, 0]
        if np.random.rand() > 0.5:
             xyz[:, 1] = -xyz[:, 1]

        scale = np.random.uniform(0.9, 1.1)
        xyz = xyz * scale
        xyz += np.random.normal(0, 0.01, xyz.shape)
        
        if pts.shape[1] > 3:
            return np.concatenate([xyz, pts[:, 3:]], axis=1)
        else:
            return xyz

    def __getitem__(self, idx):
        try:
            pts, sem = safe_load_npz(self.files[idx])
        except Exception as e:
            print(f"Error loading {self.files[idx]}: {e}. Returning zeros.")
            pts = np.zeros((self.num_points, INPUT_FEATURES), dtype=np.float32)
            sem = np.zeros(self.num_points, dtype=np.int64)
            return torch.from_numpy(pts), torch.from_numpy(sem)

        sem = align_labels(pts, sem)
        N = pts.shape[0]

        if N == 0:
            pts_s = np.zeros((self.num_points, INPUT_FEATURES), dtype=np.float32)
            mapped = np.zeros(self.num_points, dtype=np.int64)
            return torch.from_numpy(pts_s), torch.from_numpy(mapped)

        choice = np.random.choice(N, self.num_points, replace=(N < self.num_points))
        
        if pts.shape[1] < INPUT_FEATURES:
            print(f"FATAL ERROR: File {self.files[idx]} has {pts.shape[1]} features, but {INPUT_FEATURES} were requested.")
            pts_s = np.zeros((self.num_points, INPUT_FEATURES), dtype=np.float32)
            mapped = np.zeros(self.num_points, dtype=np.int64)
            return torch.from_numpy(pts_s), torch.from_numpy(mapped)
        
        pts_s_raw = pts[choice, :INPUT_FEATURES]
        sem_s_raw = sem[choice]

        xyz = pts_s_raw[:, :3]
        xyz_centered = xyz - np.mean(xyz, axis=0)
        
        max_dist = np.max(np.sqrt(np.sum(xyz_centered**2, axis=1)))
        if max_dist < 1e-6: max_dist = 1.0
        
        pts_s = xyz_centered / max_dist
        
        if self.augment: 
            pts_s = self._augment(pts_s)
        
        mapped = np.array([self.raw_to_final_idx.get(int(x), 0) for x in sem_s_raw], dtype=np.int64)
        
        return torch.from_numpy(pts_s.astype(np.float32)), torch.from_numpy(mapped)


# ---------------- NEW: DGCNN Model ----------------

def knn(x, k):
    """(B, C, N), k -> (B, N, k)"""
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    
    # (B, N, k)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

def get_graph_feature(x, k=20, idx=None):
    """(B, C, N) -> (B, 2*C, N, k)"""
    batch_size, num_dims, num_points = x.shape
    x = x.view(batch_size, -1, num_points)
    if idx is None:
        idx = knn(x, k=k) # (B, N, k)
    
    device = x.device
    idx_base = torch.arange(0, batch_size, device=device).view(-1, 1, 1) * num_points
    idx = idx + idx_base
    idx = idx.view(-1)

    x_neighbors = x.transpose(2, 1).reshape(-1, num_dims)[idx, :] # (B*N*k, C)
    x_neighbors = x_neighbors.view(batch_size, num_points, k, num_dims) # (B, N, k, C)
    
    x = x.transpose(2, 1).view(batch_size, num_points, 1, num_dims).repeat(1, 1, k, 1) # (B, N, k, C)
    
    # (B, N, k, 2*C)
    feature = torch.cat((x_neighbors - x, x), dim=3).permute(0, 3, 1, 2) # (B, 2*C, N, k)
    
    return feature

class DGCNNSeg(nn.Module):
    def __init__(self, num_classes, input_features=3, k=20):
        super(DGCNNSeg, self).__init__()
        self.k = k
        self.k_input_features = input_features * 2
        
        # Encoder
        self.conv1 = nn.Sequential(nn.Conv2d(self.k_input_features, 64, kernel_size=1, bias=False),
                                   nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(64*2, 64, kernel_size=1, bias=False),
                                   nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(64*2, 128, kernel_size=1, bias=False),
                                   nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(128*2, 256, kernel_size=1, bias=False),
                                   nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        
        # Aggregator
        self.conv5 = nn.Sequential(nn.Conv1d(64+64+128+256, 1024, kernel_size=1, bias=False),
                                   nn.BatchNorm1d(1024), nn.LeakyReLU(0.2))

        # Decoder (Segmentation Head)
        self.seg_head = nn.Sequential(
            nn.Conv1d(1024 + 1024, 512, 1, bias=False), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Conv1d(512, 256, 1, bias=False), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Conv1d(256, 128, 1, bias=False), nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
            nn.Conv1d(128, num_classes, 1)
        )

    def forward(self, x):
        # x: (B, N, C_in)
        x = x.transpose(2, 1) # (B, C_in, N)
        B, C, N = x.shape

        # --- Encoder ---
        x1 = get_graph_feature(x, k=self.k)       # (B, 2*C, N, k)
        x1 = self.conv1(x1)                       # (B, 64, N, k)
        x1 = x1.max(dim=-1, keepdim=False)[0]     # (B, 64, N)

        x2 = get_graph_feature(x1, k=self.k)      # (B, 128, N, k)
        x2 = self.conv2(x2)                       # (B, 64, N, k)
        x2 = x2.max(dim=-1, keepdim=False)[0]     # (B, 64, N)

        x3 = get_graph_feature(x2, k=self.k)      # (B, 128, N, k)
        x3 = self.conv3(x3)                       # (B, 128, N, k)
        x3 = x3.max(dim=-1, keepdim=False)[0]     # (B, 128, N)

        x4 = get_graph_feature(x3, k=self.k)      # (B, 256, N, k)
        x4 = self.conv4(x4)                       # (B, 256, N, k)
        x4 = x4.max(dim=-1, keepdim=False)[0]     # (B, 256, N)

        # --- Aggregation ---
        x_local = torch.cat((x1, x2, x3, x4), dim=1) # (B, 512, N)
        x_local = self.conv5(x_local)                # (B, 1024, N)

        # --- Global Max Pool ---
        x_global = x_local.max(dim=-1, keepdim=True)[0] # (B, 1024, 1)
        x_global_repeat = x_global.repeat(1, 1, N)      # (B, 1024, N)
        
        # --- Decoder ---
        x = torch.cat((x_local, x_global_repeat), dim=1) # (B, 2048, N)
        
        x = self.seg_head(x) # (B, num_classes, N)
        x = x.transpose(2, 1) # (B, N, num_classes)
        
        return x # Only return scores

# ---------------- Visualization ----------------
def colormap_for_labels(labels):
    num_classes = max(labels) + 1 if len(labels) > 0 else 1
    cmap = plt.get_cmap("tab20", num_classes)
    return cmap(labels % cmap.N)[:,:3]

def visualize(points, gt, pred, out_path):
    xyz = points[:, :3]
    fig = plt.figure(figsize=(12,6))
    ax1 = fig.add_subplot(121, projection="3d")
    ax1.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=colormap_for_labels(gt), s=1.0)
    ax1.set_title("Ground Truth"); ax1.set_axis_off()
    ax2 = fig.add_subplot(122, projection="3d")
    ax2.scatter(xyz[:,0], xyz[:,1], xyz[:,2], c=colormap_for_labels(pred), s=1.0)
    ax2.set_title("Prediction"); ax2.set_axis_off()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

# ---------------- Training + Simulation ----------------
def train_and_simulate():
    files = find_npz_files(DATA_ROOT)
    if not files:
        print(f"❌ Error: No .npz files found in {DATA_ROOT}. Please check the path.")
        return
        
    train_files, test_files = train_test_split(files, test_size=0.2, random_state=42)
    train_files, val_files = train_test_split(train_files, test_size=0.2, random_state=42)
    
    train_ds = NPZSegDataset(train_files, augment=True)
    val_ds = NPZSegDataset(val_files, augment=False)
    
    num_classes = train_ds.num_classes
    final_idx_to_name = train_ds.final_idx_to_name
    print(f"✅ Found {len(files)} files. Training on {num_classes-1} classes (+1 for unknown).")

    train_ds_for_weights = NPZSegDataset(train_files, augment=False)
    train_loader_weights = DataLoader(train_ds_for_weights, batch_size=BATCH_SIZE * 4, num_workers=0) 
    class_weights = calculate_class_weights(train_loader_weights, num_classes).to(DEVICE)
    print("Class weights calculated and clipped:", class_weights)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=0)

    # --- CHANGED ---: Using new DGCNN model
    model = DGCNNSeg(num_classes, input_features=INPUT_FEATURES).to(DEVICE)
    
    opt = optim.AdamW(model.parameters(), lr=LR)
    
    # --- We MUST ignore index 0 (our "unknown" class)
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=0)
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    stats = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
    best_path = os.path.join(OUT_ROOT, "dgcnn_simplified_best.pth")
    best_val_loss = float('inf')

    # -------- Training --------
    for epoch in range(1, EPOCHS + 1):
        model.train(); tloss=tacc=0; t_total=0
        for pts, labels in tqdm(train_loader, desc=f"Train {epoch}"):
            pts, labels = pts.to(DEVICE).float(), labels.to(DEVICE).long()
            
            if pts.shape[0] == 0: continue

            opt.zero_grad()
            
            # --- CHANGED ---: Model now only returns scores
            out = model(pts)
            B, N, C = out.shape
            
            # --- CHANGED ---: No more T-Net loss
            loss = criterion(out.reshape(-1, C), labels.reshape(-1))
            
            if torch.isnan(loss) or torch.isinf(loss):
                print("⚠️ NaN or Inf loss detected! Skipping batch.")
                continue

            loss.backward(); opt.step()
            
            preds = out.argmax(2)
            
            valid_mask = (labels != 0)
            if valid_mask.sum() > 0:
                acc = (preds[valid_mask] == labels[valid_mask]).float().mean().item() * 100
                tacc += acc
                t_total += 1
                
            tloss += loss.item()
        
        train_loss = tloss / len(train_loader)
        train_acc = tacc / t_total if t_total > 0 else 0.0
        scheduler.step()

        model.eval(); vloss=vacc=0; v_total=0
        with torch.no_grad():
            for pts, labels in val_loader:
                pts, labels = pts.to(DEVICE).float(), labels.to(DEVICE).long()
                
                if pts.shape[0] == 0: continue

                out = model(pts); B, N, C = out.shape
                loss = criterion(out.reshape(-1, C), labels.reshape(-1))
                
                preds = out.argmax(2)
                
                valid_mask = (labels != 0)
                if valid_mask.sum() > 0:
                    acc = (preds[valid_mask] == labels[valid_mask]).float().mean().item() * 100
                    vacc += acc
                    v_total += 1

                vloss += loss.item()
                
        val_loss = vloss / len(val_loader)
        val_acc = vacc / v_total if v_total > 0 else 0.0
        
        print(f"Epoch {epoch}: Train {train_loss:.4f}/{train_acc:.1f}% | Val {val_loss:.4f}/{val_acc:.1f}% | LR {opt.param_groups[0]['lr']:.2e}")
        stats["train_loss"].append(train_loss); stats["val_loss"].append(val_loss)
        stats["train_acc"].append(train_acc); stats["val_acc"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_path)
            print(f"✅ New best model saved with validation loss: {val_loss:.4f}")

    # -------- Save Training Curves --------
    plt.figure()
    plt.plot(stats["train_loss"], label="Train Loss")
    plt.plot(stats["val_loss"], label="Val Loss")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.savefig(os.path.join(OUT_ROOT, "loss_curve.png")); plt.close()

    plt.figure()
    plt.plot(stats["train_acc"], label="Train Acc")
    plt.plot(stats["val_acc"], label="Val Acc")
    plt.legend(); plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)")
    plt.savefig(os.path.join(OUT_ROOT, "acc_curve.png")); plt.close()
    print("📊 Training curves saved.")

    # -------- Load Best Model for Simulation --------
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        print("✅ Best model loaded for simulation.")
    else:
        print("⚠️ No best model was saved. Running simulation with the last model state.")

    # -------- Simulation Loop --------
    sim_ds = val_ds
    model.eval()
    simulation_files = random.sample(sim_ds.files, min(len(sim_ds.files), NUM_VIS_SAMPLES))
    
    for i, sample_file in enumerate(simulation_files):
        try:
            file_idx = sim_ds.files.index(sample_file)
            pts_norm, sem_mapped = sim_ds[file_idx]
            pts_norm = pts_norm.unsqueeze(0).to(DEVICE)
            sem_s = sem_mapped.numpy()
        except Exception as e:
            print(f"Could not load or process simulation file {sample_file}: {e}")
            continue

        with torch.no_grad():
            out = model(pts_norm) # DGCNN only returns scores
            pred = out.argmax(2).squeeze(0).cpu().numpy()

        counts = Counter(pred)
        print(f"\n🟢 Detected Objects (Sample {i+1}):")
        
        detected_danger = False
        danger_messages = []
        for idx, c in counts.items():
            name = final_idx_to_name.get(idx, "unknown")
            print(f" - {name}: {c} points")
            if name in danger_objs:
                detected_danger = True
                danger_messages.append(name)
                print(f"🚨 DANGER: {name.upper()} DETECTED! 🚨")
        
        if detected_danger:
            print("🔊 HAPTIC FEEDBACK: VIBRATION ALERT")
            try:
                engine = pyttsx3.init()
                message = f"Warning, a {', '.join(danger_messages)} was detected ahead."
                engine.say(message)
                engine.runAndWait()
            except Exception as e:
                print(f"Could not initialize text-to-speech engine: {e}")

        visualize(pts_norm.squeeze(0).cpu().numpy(), sem_s, pred, os.path.join(OUT_ROOT, f"simulation_{i+1}.png"))
        print(f"🖼️ Simulation {i+1} visualization saved.")

if __name__ == "__main__":
    # Note: num_workers=0 is set in the DataLoaders to ensure compatibility
    # with Jupyter notebooks and Windows.
    train_and_simulate()

Device: cuda
✅ Found 3914 files. Training on 12 classes (+1 for unknown).


Calculating class weights: 100%|██████████| 40/40 [01:09<00:00,  1.73s/it]


Class weights calculated and clipped: tensor([1.0000e+00, 1.0000e+03, 1.0000e+03, 1.0000e+03, 1.9623e+00, 1.1884e-01,
        1.0000e+03, 7.9830e+01, 2.5360e+02, 8.7842e-01, 1.0405e+02, 3.5901e-01,
        7.9469e+00], device='cuda:0')


Train 1: 100%|██████████| 157/157 [07:03<00:00,  2.70s/it]


Epoch 1: Train 2.0302/33.4% | Val 1.9155/48.1% | LR 1.00e-04
✅ New best model saved with validation loss: 1.9155


...

Epoch 98: Train 0.8269/54.9% | Val 0.8555/60.1% | LR 1.10e-06


Train 99: 100%|██████████| 157/157 [06:48<00:00,  2.60s/it]


Epoch 99: Train 0.8526/56.1% | Val 0.8499/62.9% | LR 1.02e-06


Train 100: 100%|██████████| 157/157 [06:54<00:00,  2.64s/it]


Epoch 100: Train 0.8359/55.2% | Val 0.8355/59.6% | LR 1.00e-06
📊 Training curves saved.
✅ Best model loaded for simulation.

🟢 Detected Objects (Sample 1):
 - door: 1024 points
🖼️ Simulation 1 visualization saved.

🟢 Detected Objects (Sample 2):
 - wall: 1024 points
🖼️ Simulation 2 visualization saved.

🟢 Detected Objects (Sample 3):
 - chair: 953 points
 - door: 71 points
🖼️ Simulation 3 visualization saved.

🟢 Detected Objects (Sample 4):
 - table: 907 points
 - window: 115 points
 - door: 2 points
🖼️ Simulation 4 visualization saved.

🟢 Detected Objects (Sample 5):
 - door: 447 points
 - chair: 559 points
 - window: 18 points
🖼️ Simulation 5 visualization saved.
